# CircuitVision • Export Sam's YOLO11s specialist model

This notebook converts your **existing 10-class Sam checkpoint** into CPU-compatible ONNX for our website. **This is not training, does not merge the models, and does not change the live website.** CPU runtime is sufficient.

Use only Sam's original `my_model(1).pt` (19 MB) when prompted; we pin its SHA256. Upload the resulting `Sam_Specialist_ONNX.zip` into our chat for inspection. No API keys required.

The Sam checkpoint class labels are **BUZZER, CERAMIC CAPACITOR, DIODE, LED (BLUE), LED (GREEN), POTENTIOMETER, POWER MODULE, RESISTOR 10K, RESISTOR 2K, RESISTOR 52K**. These are classification labels, not measured electrical values.

## 1. Install tools

In [ ]:
%pip -q install "ultralytics>=8.3,<9" "onnx>=1.15,<2" "onnxruntime>=1.18,<2"
import hashlib, json, shutil, zipfile
from pathlib import Path
from google.colab import files
import numpy as np
import onnxruntime as ort
from ultralytics import YOLO
print("Ready; GPU is optional")

## 2. Upload exact model and verify its checksum before loading

In [ ]:
SHA='9528dd6c67291abc6c99e22698521b1a3e436ef516dc7239777275ed91c66b5d'
EXPECTED=['BUZZER','CERAMIC CAPACITOR','DIODE','LED (BLUE)','LED (GREEN)','POTENTIOMETER','POWER MODULE','RESISTOR 10K','RESISTOR 2K','RESISTOR 52K']
uploaded=files.upload()
assert len(uploaded)==1,'Upload exactly one checkpoint'
source=Path('/content')/next(iter(uploaded))
assert hashlib.sha256(source.read_bytes()).hexdigest()==SHA,'Wrong checkpoint: stop.'
print('Verified:',source.name,source.stat().st_size,'bytes')

## 3. Export with ten original classes, static 640 input, no baked-in NMS

In [ ]:
model=YOLO(str(source))
classes=[str(model.names[i]) for i in range(len(model.names))]
assert classes==EXPECTED,'Sam model class IDs differ from verified checkpoint'
out=Path(model.export(format='onnx',imgsz=640,opset=17,dynamic=False,nms=False,simplify=False))
assert out.exists() and out.stat().st_size>1000000
session=ort.InferenceSession(str(out),providers=['CPUExecutionProvider'])
inp=session.get_inputs()[0]
assert list(inp.shape)==[1,3,640,640]
assert 14 in list(session.get_outputs()[0].shape),'Expected 4 box channels + 10 labels'
test=session.run(None,{inp.name:np.full((1,3,640,640),.4,dtype=np.float32)})
assert len(test)==1 and np.all(np.isfinite(test[0]))
print('CPU ONNX model interface passes synthetic smoke test')

## 4. Package and download for Rad's inspection

In [ ]:
folder=Path('/content/sam_specialist_package')
folder.mkdir(exist_ok=True)
shutil.copy2(out,folder/'sam_10.onnx')
(folder/'sam_classes.json').write_text(json.dumps({str(i):label for i,label in enumerate(classes)},indent=2))
(folder/'SAM_MODEL_REPORT.json').write_text(json.dumps({'source_sha256':SHA,'architecture':'YOLO11s','class_count':10,'classes':classes,'train_performed':False,'dataset_available':True,'value_predictions_are_measurements':False},indent=2))
archive=Path('/content/Sam_Specialist_ONNX.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for item in sorted(folder.iterdir()):z.write(item,arcname=item.name)
print('ZIP SHA256:',hashlib.sha256(archive.read_bytes()).hexdigest())
files.download(str(archive))

**Next:** Upload `Sam_Specialist_ONNX.zip` here. I'll verify its files and prepare a conservative integration that only offers value-specific **hints** when Sam and our existing detector agree. Sam's original `MyDrive/YOLO/dataset/data.zip` remains necessary to retrain and independently evaluate a single truly combined model.